In [1]:
#使用Sequence
import gradio as gr
import pydicom
import cv2
import hashlib
from pydicom.datadict import tag_for_keyword
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding
from pydicom.uid import ExplicitVRLittleEndian
import io
from pydicom.dataset import Dataset
from pydicom import Sequence
from datetime import datetime


def create_signature_item(signature, signature_time, public_key, auth_type, auth_fields, auth_region):
    """
    创建一个包含签名信息的数据集。
    
    参数:
    signature (str): 数据签名
    signature_time (str): 数据签名时间
    public_key (str): 公钥 (PEM格式)
    auth_type (int): 认证对象类型 (0: IMAGE Pixel, 1: Image Attribute)
    auth_fields (str): 认证的图像属性字段列表
    auth_region (str): 认证图像区域 (UL, UR, BL, BR)

    返回:
    Dataset: 包含签名信息的数据集
    """
    item = Dataset()
    item.add_new((0x9001, 0x0001), 'LO', signature)
    item.add_new((0x9001, 0x0002), 'LO', signature_time)
    item.add_new((0x9001, 0x0003), 'LO', public_key)
    item.add_new((0x9001, 0x0004), 'IS', str(auth_type))
    item.add_new((0x9001, 0x0005), 'LO', auth_fields)
    item.add_new((0x9001, 0x0006), 'LO', auth_region)
    return item



# Create a Gradio interface for uploading DICOM files
def upload_dicom_file(file,sopInsUID):
    # Load the uploaded DICOM file
    dicom_data = pydicom.dcmread(file,force=True)
    dicom_file = pydicom.dcmread(file,force=True)
    dicom_data.is_implicit_VR = False
    
    # 私钥和公钥
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
    )
    public_key1 = private_key.public_key()
    #私钥
    private_pem = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.TraditionalOpenSSL,
    encryption_algorithm=serialization.NoEncryption()
    )

    # 公钥
    public_pem = public_key1.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
    
    )
    with open("D:\\Che Qiancao\\sjtu\\课程\\7大四上\\课程设计\\lesson4\\private_key.pem", "wb") as private_file:
        private_file.write(private_pem)
        
    with open("D:\\Che Qiancao\\sjtu\\课程\\7大四上\\课程设计\\lesson4\\public_key.pem", "wb") as public_file:
        public_file.write(public_pem)
#图像加密
    image_data = dicom_data.pixel_array  
    gray_image_data = cv2.cvtColor(image_data, cv2.COLOR_BGR2GRAY)
    height, width = gray_image_data.shape
    
    top_left = image_data[:height // 2, :width // 2]
    top_right = image_data[:height // 2, width // 2:]
    bottom_left = image_data[height // 2:, :width // 2]
    bottom_right = image_data[height // 2:, width // 2:]
    
    #加密并且生成signature，生成时间
    def sig_dicom(dicom_image):
        #hash 值
        sha256 = hashlib.sha256()
        sha256.update(dicom_image.tobytes())
        hash_value=sha256.digest()
        #加密
        encrypted_hash = private_key.sign(
            hash_value,
            padding.PKCS1v15(),
            hashes.SHA256()
        )
        current_time = datetime.now().isoformat()
        time_sig = private_key.sign(
            current_time.encode(),
            padding.PKCS1v15(),
            hashes.SHA256()
    )
        return encrypted_hash, time_sig
    
    
    sig_top_left, time_top_left=sig_dicom(top_left)
    sig_top_right, time_top_right=sig_dicom(top_right)
    sig_bottom_left, time_bottom_left=sig_dicom(bottom_left)
    sig_bottom_right, time_bottom_right=sig_dicom(bottom_right)
    public_key1=public_pem
    dicom_type0=0 #图像
    dicom_type1=1 #病人信息
    inform0=0
    
    
    ds = pydicom.dcmread(file,force=True)
    
    # 创建序列
    
    #病人信息
    patient_name = dicom_data.PatientName
    patient_age = dicom_data.PatientAge


    metadata_str = f"{patient_name} {patient_age}"
    meta_sig = private_key.sign(
            metadata_str.encode(),
            padding.PKCS1v15(),
            hashes.SHA256()
    )
    current_time = datetime.now().isoformat()
    time_meta = private_key.sign(
        current_time.encode(),
        padding.PKCS1v15(),
        hashes.SHA256()
    )
    # 添加五个示例签名
    signatures_sequence = []
    
    # UL 区域
    ul_signature_dataset = create_signature_item(
        signature=sig_top_left,
        signature_time=time_top_left,
        public_key=public_key1,
        auth_type='0',
        auth_fields='',
        auth_region='UL'
    )
    signatures_sequence.append(ul_signature_dataset)
    
    # UR 区域
    ur_signature_dataset = create_signature_item(
        signature=sig_top_right,
        signature_time=time_top_right,
        public_key=public_key1,
        auth_type='0',
        auth_fields='',
        auth_region='UR'
    )
    signatures_sequence.append(ur_signature_dataset)
    
    # BL 区域
    bl_signature_dataset = create_signature_item(
        signature=sig_bottom_left,
        signature_time=time_bottom_left,
        public_key=public_key1,
        auth_type='0',
        auth_fields='',
        auth_region='BL'
    )
    signatures_sequence.append(bl_signature_dataset)
    
    # BR 区域
    br_signature_dataset = create_signature_item(
        signature=sig_bottom_right,
        signature_time=time_bottom_right,
        public_key=public_key1,
        auth_type='0',
        auth_fields='',
        auth_region='BR'
    )
    signatures_sequence.append(br_signature_dataset)
    
    # Attributes 区域
    attributes_signature_dataset = create_signature_item(
        signature=meta_sig,
        signature_time=time_meta,
        public_key=public_key1,
        auth_type='1',
        auth_fields='PatientName, PatientAge',
        auth_region='Attributes'
    )
    signatures_sequence.append(attributes_signature_dataset)
    
    dicom_file.add_new((0x9001, 0x1000), 'SQ', signatures_sequence)

    # 设置传输语法为显式VR小端
    dicom_file.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    # 确保使用正确的传输语法保存文件
    dicom_file.is_implicit_VR = False
    dicom_file.is_little_endian = True

    # 保存DICOM文件为显式VR格式
    new_file_path = r'D:\Che Qiancao\sjtu\课程\7大四上\课程设计\项目设计_模块3_520021910466_仇清扬\\' + sopInsUID + 'modified_dicom_file1.dcm'  # 新文件的路径
    dicom_file.save_as(new_file_path, write_like_original=False)
    print(dicom_file)
    
    dicom_verified="D:\Che Qiancao\sjtu\课程\7大四上\课程设计\项目设计_模块3_520021910466_仇清扬\\"+ sopInsUID + "modified_dicom_file1.dcm"
    return dicom_verified


In [2]:
import sys
import os
from PyQt5.QtCore import QTimer, QDateTime, Qt, pyqtSignal
from PyQt5.QtGui import QFont, QImage, QPixmap
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QVBoxLayout, QHBoxLayout, QPushButton, QDialog, QFormLayout, QLineEdit, QRadioButton
import configparser
from pydicom.dataset import FileMetaDataset
from pydicom.uid import UID
from pydicom import Dataset
import pydicom

import cv2
import numpy as np
import sqlite3
import time
import random
from datetime import datetime

class PatientInfoDialog(QDialog):
    patient_info_signal = pyqtSignal(str, str, str, str, str)

    def __init__(self):
        super(PatientInfoDialog, self).__init__()

        self.setWindowTitle("Patient Information")
        self.setGeometry(100, 200, 300, 200)

        self.fbox = QFormLayout(self)

        self.name_label = QLabel("Name")
        self.name_edit = QLineEdit()
        self.fbox.addRow(self.name_label, self.name_edit)

        self.age_label = QLabel("Age")
        self.age_edit = QLineEdit()
        self.fbox.addRow(self.age_label, self.age_edit)

        self.sex_label = QLabel("Sex")
        self.sex_hbox = QHBoxLayout()
        self.male_radio = QRadioButton("Male")
        self.female_radio = QRadioButton("Female")
        self.sex_hbox.addWidget(self.male_radio)
        self.sex_hbox.addWidget(self.female_radio)
        self.fbox.addRow(self.sex_label, self.sex_hbox)

        self.physician_label = QLabel("Physician")
        self.physician_edit = QLineEdit()
        self.fbox.addRow(self.physician_label, self.physician_edit)

        self.comments_label = QLabel("Comments")
        self.comments_edit = QLineEdit()
        self.fbox.addRow(self.comments_label, self.comments_edit)

        submit_button = QPushButton("Submit")
        submit_button.clicked.connect(self.accept)
        cancel_button = QPushButton("Cancel")
        cancel_button.clicked.connect(self.reject)
        self.fbox.addRow(submit_button, cancel_button)

    def accept(self):
        # 发送信号，传递患者信息
        name = self.name_edit.text()
        sex = "Male" if self.male_radio.isChecked() else "Female"
        age = self.age_edit.text()
        physician = self.physician_edit.text()
        comments = self.comments_edit.text()
        self.patient_info_signal.emit(name, sex, age, physician, comments)
        super().accept()

In [3]:
class MainUI(QWidget):
    def __init__(self):
        super(MainUI, self).__init__()
        self.ImaNum = 0
        self.setGeometry(100, 200, 800, 800)
        self.setWindowTitle('PyQt5')
        self.setStyleSheet("background-color: black;")

        layout = QVBoxLayout(self)
        layout.setAlignment(Qt.AlignTop)

        font = QFont()
        font.setFamily("Arial")
        font.setPointSize(25)

        margin = 100

        self.name_label = QLabel("<a href='#'>Name: XXX</a>", self)
        self.name_label.setStyleSheet("color: white;")
        self.name_label.setFont(font)
        self.name_label.setMargin(margin)
        self.name_label.linkActivated.connect(self.show_patient_info_dialog)

        self.gender_label = QLabel("Sex: M\nAge: 41", self)
        self.gender_label.setStyleSheet("color: white;")
        self.gender_label.setFont(font)
        self.gender_label.setMargin(margin)

        self.datetime_label = QLabel()
        self.datetime_label.setStyleSheet("color: white;")
        self.datetime_label.setFont(font)
        self.datetime_label.setMargin(margin)

        self.physician_label = QLabel("PHY:\nCMT:", self)
        self.physician_label.setStyleSheet("color: white;")
        self.physician_label.setFont(font)
        self.physician_label.setMargin(margin)

        self.comments_label = QLabel("", self)
        self.comments_label.setStyleSheet("color: white;")
        self.comments_label.setFont(font)
        self.comments_label.setMargin(margin)
        
        lbl_Image = QLabel(self)
        self.image_label = lbl_Image

        left_layout = QVBoxLayout()
        left_layout.addWidget(self.name_label)
        left_layout.addWidget(self.gender_label)
        left_layout.addWidget(self.datetime_label)
        left_layout.addWidget(self.physician_label)
        left_layout.addWidget(self.comments_label)

        right_layout = QHBoxLayout()
        right_layout.addStretch(1)
        right_layout.addWidget(lbl_Image)

        combined_layout = QHBoxLayout(self)
        combined_layout.addLayout(left_layout)
        combined_layout.addLayout(right_layout)

        layout.addLayout(combined_layout)

        timer_datetime = QTimer(self)
        timer_datetime.timeout.connect(self.update_datetime)
        timer_datetime.start(1000)

        timer_image = QTimer(self)
        timer_image.timeout.connect(self.update_image)
        timer_image.start(int(1000 / 30))

        self.update_datetime()

        self.cap = cv2.VideoCapture(0)
        self.cap.set(3, 1280)
        self.cap.set(4, 720)

        self.config = configparser.ConfigParser()
        self.config.read('config.ini')
        self.mask_shape = self.config.get('SystemConfig', 'OverlayShape')
        self.image_storage_path = self.config.get('SystemConfig', 'ImageStoragePath')

        self.dialog = None
        self.image_sop_instance = {'Image': {'SOPInsUID': None}}
    

    def update_datetime(self):
        current_datetime = QDateTime.currentDateTime()
        date = current_datetime.toString('yyyy/MM/dd')
        time = current_datetime.toString('hh:mm:ss')
        self.datetime_label.setText(f"{date}\n{time}")

    def update_image(self):
        if self.image_label and self.cap.isOpened():
            ret, frame = self.cap.read()
            if ret:
                frame_rgbb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mask = self.create_mask(frame_rgbb.shape, self.mask_shape)
                frame_rgb = cv2.bitwise_and(frame_rgbb, frame_rgbb, mask=mask)
                height, width, channel = frame_rgb.shape
                bytes_per_line = 3 * width
                q_image = QImage(frame_rgb.data, width, height, bytes_per_line, QImage.Format_RGB888)
                pixmap = QPixmap.fromImage(q_image)
                self.image_label.setPixmap(pixmap)

    def create_mask(self, shape, mask_shape):
        if mask_shape == 'Circles':
            center = (shape[1] // 2, shape[0] // 2)
            radius = min(center)
            mask = np.zeros(shape[:2], dtype=np.uint8)
            cv2.circle(mask, center, radius, (255, 255, 255), -1)
        elif mask_shape == 'Squares':
            mask = np.ones(shape[:2], dtype=np.uint8) * 255
        elif mask_shape == 'Octagons':
            mask = np.zeros(shape[:2], dtype=np.uint8)
            center = (shape[1] // 2, shape[0] // 2)
            side_length = min(shape[1], shape[0]) // 2
            octagon_pts = self.generate_octagon(center, side_length)
            cv2.fillPoly(mask, [octagon_pts], 255)

        return mask

    def generate_octagon(self, center, side_length):
        angles = np.linspace(0, 2 * np.pi, 8, endpoint=False)
        octagon_pts = np.array(
            [(center[0] + int(np.cos(angle) * side_length), center[1] + int(np.sin(angle) * side_length))
             for angle in angles], dtype=np.int32)
        return octagon_pts

    def show_patient_info_dialog(self):
        image_sop_instance = {
            'Patient': {'PatNam': '', 'Sex': '', 'Age': ''},
            'Study': {'StuInsUID': '', 'OtherStudyInfo': ''},
            'Series': {'SerInsUID': '', 'OtherSeriesInfo': ''},
            'Image': {'SOPInsUID': '', 'StoragePath': ''}
        }

        success = self.openPatientInfoDialog(image_sop_instance)

        if success:
            self.updatePatientInfoUI(image_sop_instance)
            self.image_sop_instance = image_sop_instance
            self.captureAndSaveImage(self.image_sop_instance)

    def openPatientInfoDialog(self, image_sop_instance):
        self.dialog = PatientInfoDialog()

        def process_and_update(name, sex, age, physician, comments):
            self.processPatientInfo(name, sex, age, physician, comments, image_sop_instance)

            # Fill image_sop_instance with patient, Study, Series information
            image_sop_instance["Patient"] = {
                "PatNam": name,
                "PatBirDate": self.age_to_birthdate(int(age)),
                #TODO 字典的问题
                "PatSex": "M" if sex == "Male" else "F",
                "Sex": "M" if sex == "Male" else "F",
                "Age": int(age),
                "NumPatRelStu": 0,
                "NumPatRelSer": 0,
                "NumPatRelIma": 0
            }
            # TODO
            # image_sop_instance["Study"] ={
            #     "StuID": 1,
            #     "StuDate": datetime.now().strftime('%Y%m%d'),
            #     "StuTime": datetime.now().strftime('%H%M%S'),
            #     "AccNum": "1",
            #     "PatAge": age,
            #     "PatSize": 170,
            #     "PatWeight": 70,
            #     "NumStuRelSer": 0,
            #     "NumStuRelIma": 0,
            # }

            image_sop_instance["Study"].update({
                "StuID": 1,
                "StuDate": datetime.now().strftime('%Y%m%d'),
                "StuTime": datetime.now().strftime('%H%M%S'),
                "AccNum": "1",
                "PatAge": age,
                "PatSize": 170,
                "PatWeight": 70,
                "NumStuRelSer": 0,
                "NumStuRelIma": 0,
            })

            image_sop_instance["Series"].update({
                "SerNum": "1",
                "Modality": "ES",
                "ProNam": physician,
                "SerDes": comments,
                "BodParExa": "stomach",
                "NumSerRelIma": 0
            })

            # Generate unique IDs using the existing function generateUniqueID()
            image_sop_instance["Patient"]["PatID"] = self.generateUniqueID("2.25.", "PatID")
            image_sop_instance["Study"]["StuInsUID"] = self.generateUniqueID("2.25.", "StudyInsUID")
            image_sop_instance["Series"]["SerInsUID"] = self.generateUniqueID("2.25.", "SerInsUID")

            print(image_sop_instance)

        self.dialog.patient_info_signal.connect(process_and_update)
        result = self.dialog.exec_()

        return result == QDialog.Accepted
    
    def processPatientInfo(self, name, sex, age, physician, comments, image_sop_instance):
        prefix = "2.25."

        image_sop_instance['Patient']['PatID'] = self.generateUniqueID(prefix, "PatID")
        image_sop_instance['Study']['StuInsUID'] = self.generateUniqueID(prefix, "StudyInsUID")
        image_sop_instance['Series']['SerInsUID'] = self.generateUniqueID(prefix, "SerInsUID")

        image_sop_instance['Patient']['PatNam'] = name
        image_sop_instance['Patient']['Sex'] = sex
        image_sop_instance['Patient']['Age'] = age
        image_sop_instance['Study']['OtherStudyInfo'] = f"Physician: {physician}"
        image_sop_instance['Series']['OtherSeriesInfo'] = f"Comments: {comments}"

        self.updatePatientInfoUI(image_sop_instance)

    def captureAndSaveImage(self, image_sop_instance):
        ret,capturedImage = self.cap.read()
        if ret:
            self.ImaNum = self.ImaNum + 1 
        sopInsUID = self.generateUniqueID("2.25.", "ImageSOPInsUID")
        self.image_sop_instance.setdefault('Image', {})['SOPInsUID'] = sopInsUID
        self.image_sop_instance.setdefault('Image', {})['ImaNum'] = self.ImaNum
        self.image_sop_instance.setdefault('Image', {})['SOPClaUID'] = pydicom.uid.SecondaryCaptureImageStorage
        self.image_sop_instance.setdefault('Image', {})['TransferSyntax'] = pydicom.uid.ExplicitVRLittleEndian

        storagePath = "D:\\Che Qiancao\\sjtu\\课程\\7大四上\\课程设计\\项目设计_模块3_520021910466_仇清扬\\" + sopInsUID + ".dcm"
        dicomImage = self.encodeToDICOM(image_sop_instance, capturedImage)
        self.saveDICOMImage(dicomImage, storagePath)
        image_sop_instance['Image']['StoragePath'] = storagePath
        self.updateDatabaseWithImageInfo(image_sop_instance)
        upload_dicom_file("D:\\Che Qiancao\\sjtu\\课程\\7大四上\\课程设计\\项目设计_模块3_520021910466_仇清扬\\" + sopInsUID + ".dcm",sopInsUID)
        
    def encodeToDICOM(self, image_sop_instance, capturedImage):
        dicom_file = Dataset()
        file_meta = FileMetaDataset()
        file_meta.MediaStorageSOPClassUID = pydicom.uid.SecondaryCaptureImageStorage
        file_meta.MediaStorageSOPInstanceUID = image_sop_instance['Image']['SOPInsUID']
        file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian

        dicom_file.file_meta = file_meta
        dicom_file.is_little_endian = True
        dicom_file.is_implicit_VR = False

        dicom_file.PatientName = image_sop_instance['Patient']['PatNam']
        dicom_file.PatientID = image_sop_instance['Patient']['PatID']
        dicom_file.PatientBirthDate = image_sop_instance['Patient']['PatBirDate']
        dicom_file.PatientSex = image_sop_instance['Patient']['PatSex']

        dicom_file.StudyID = str(image_sop_instance['Study']['StuID'])
        dicom_file.StudyInstanceUID = image_sop_instance['Study']['StuInsUID']
        dicom_file.StudyDate = image_sop_instance['Study']['StuDate']
        dicom_file.StudyTime = image_sop_instance['Study']['StuTime']
        dicom_file.AccessionNumber = str(image_sop_instance['Study']['AccNum'])
        dicom_file.PatientAge = str(image_sop_instance['Study']['PatAge'])
        dicom_file.PatientSize = float(image_sop_instance['Study']['PatSize'])
        dicom_file.PatientWeight = float(image_sop_instance['Study']['PatWeight'])

        dicom_file.SeriesNumber = str(image_sop_instance['Series']['SerNum'])
        dicom_file.SeriesInstanceUID = image_sop_instance['Series']['SerInsUID']
        dicom_file.Modality = image_sop_instance['Series']['Modality']
        dicom_file.ProtocolName = image_sop_instance['Series']['ProNam']
        dicom_file.SeriesDescription = image_sop_instance['Series']['SerDes']
        dicom_file.BodyPartExamined = image_sop_instance['Series']['BodParExa']

        dicom_file.InstanceNumber = str(image_sop_instance['Image']['ImaNum'])
        dicom_file.SOPInstanceUID = image_sop_instance['Image']['SOPInsUID']
        dicom_file.ImageType = "DERIVED\\SECONDARY"

        dicom_file.Rows, dicom_file.Columns = capturedImage.shape[:2]
        dicom_file.PixelRepresentation = 0
        dicom_file.SamplesPerPixel = 3
        dicom_file.PhotometricInterpretation = "RGB"
        dicom_file.PlanarConfiguration = 0

        dicom_file.BitsAllocated = 8
        dicom_file.BitsStored = 8
        dicom_file.HighBit = 7

        dicom_file.PixelData = convert_image_to_dicom_format(capturedImage)

        return dicom_file

    def saveDICOMImage(self, dicomObject, storagePath):
        with open(storagePath, 'wb') as file:
            dicomObject.save_as(file)

    def updateDatabaseWithImageInfo(self, image_sop_instance):
        conn = sqlite3.connect('pcas2023.db')  # Replace with your actual database filename
        cursor = conn.cursor()
        
        cursor.execute("SELECT * FROM PatientLevel WHERE PatID = ?", (image_sop_instance['Patient']['PatID'],))
        if cursor.fetchone() is None:  #如果患者（Patient）不存在于数据库
            cursor.execute("""
                INSERT INTO PatientLevel(PatID, PatNam, PatBirDate, PatSex)
                VALUES (?, ?, ?, ?)
            """, (image_sop_instance['Patient']['PatID'], image_sop_instance['Patient']['PatNam'], 
                  image_sop_instance['Patient']['PatBirDate'], image_sop_instance['Patient']['PatSex']))
            
            # 检查并插入检查信息
        cursor.execute("SELECT * FROM StudyLevel WHERE StuInsUID = ?", (image_sop_instance['Study']['StuInsUID'],))
        if cursor.fetchone() is None: #如果检查(Study)不存在于数据库
            cursor.execute("""
                INSERT INTO StudyLevel(StuInsUID, StuID, StuDate, StuTime, AccNum, PatAge, PatSize, PatWeight, PatID)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (image_sop_instance['Study']['StuInsUID'], image_sop_instance['Study']['StuID'], 
                  image_sop_instance['Study']['StuDate'], image_sop_instance['Study']['StuTime'], 
                  image_sop_instance['Study']['AccNum'], image_sop_instance['Study']['PatAge'], 
                  image_sop_instance['Study']['PatSize'], image_sop_instance['Study']['PatWeight'], 
                  image_sop_instance['Patient']['PatID']))

    # 检查并插入序列信息 
        cursor.execute("SELECT * FROM SeriesLevel WHERE SerInsUID = ?", (image_sop_instance['Series']['SerInsUID'],))
        if cursor.fetchone() is None: #如果序列（Series）不存在于数据库
            cursor.execute("""
                INSERT INTO SeriesLevel(SerInsUID, SerNum, Modality, ProNam, SerDes, BodParExa, StuInsUID)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (image_sop_instance['Series']['SerInsUID'], image_sop_instance['Series']['SerNum'], 
                  image_sop_instance['Series']['Modality'], image_sop_instance['Series']['ProNam'], 
                  image_sop_instance['Series']['SerDes'], image_sop_instance['Series']['BodParExa'], 
                  image_sop_instance['Study']['StuInsUID']))

    # 检查并插入图像信息
        cursor.execute("SELECT * FROM ImageLevel WHERE SOPInsUID = ?", (image_sop_instance['Image']['SOPInsUID'],))
        if cursor.fetchone() is None: #如果图像（Image）不存在于数据库
            cursor.execute("""
                INSERT INTO ImageLevel(SOPInsUID, ImaNum, SOPClaUID, TransferSyntax, StoragePath, SerInsUID)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (image_sop_instance['Image']['SOPInsUID'], image_sop_instance['Image']['ImaNum'], 
                  image_sop_instance['Image']['SOPClaUID'], image_sop_instance['Image']['TransferSyntax'], 
                  image_sop_instance['Image']['StoragePath'], image_sop_instance['Series']['SerInsUID']))


        conn.commit()
        conn.close()

    def updatePatientInfoUI(self, image_sop_instance):
        self.name_label.setText(f"<a href='#'>Name: {image_sop_instance['Patient']['PatNam']}</a>")
        gender_and_age = f"Sex: {image_sop_instance['Patient']['Sex']}\nAge: {image_sop_instance['Patient']['Age']}"
        self.gender_label.setText(gender_and_age)

        physician_label_text = f"PHY: {image_sop_instance['Study']['OtherStudyInfo'].split(':')[1]}\nCMT: {image_sop_instance['Series']['OtherSeriesInfo'].split(': ')[1]}"
        self.physician_label.setText(physician_label_text)
        self.comments_label.setText(f"Comments: {image_sop_instance['Series']['OtherSeriesInfo'].split(': ')[1]}")

    def generateUniqueID(self, prefix, type):
        typeStr = ""
        if type == "PatID":
            typeStr = "1."
        elif type == "StudyInsUID":
            typeStr = "2."
        elif type == "SerInsUID":
            typeStr = "3."
        elif type == "ImageSOPInsUID":
            typeStr = "4."

        randomPart = str(int(time.time())) + str(random.randint(1, 9999))

        uid = prefix + typeStr + randomPart
        return uid
    # TODO: self arguments
    def age_to_birthdate(self, age):
        try:
            birth_year = datetime.now().year - int(age)
            return f"{birth_year}-1-1"
        except ValueError:
            return "2000-1-1"


In [4]:
def convert_image_to_dicom_format(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.reshape(-1, 3)
    image = image.flatten('C')
    return image.tobytes()

In [5]:
import threading
import time

def sendImagesToTargetSystem(sqlite_DB_Filename, destImageSys):
    try:
        # 打开数据库连接
        connection = sqlite3.connect(sqlite_DB_Filename)
        cursor = connection.cursor()

        # 检索sendFlag = 0的图像信息记录集
        cursor.execute("SELECT * FROM imageLevel WHERE sendFlag = 0")
        unSendImageRecords = cursor.fetchall()

        # 遍历unSendImageRecords
        for imageRecord in unSendImageRecords:
            # 发送DICOM图像文件
            ret = sendDICOMImage(imageRecord['D:\\Che Qiancao\\sjtu\\课程\\7大四上\\课程设计\\项目设计_模块3_520021910466_仇清扬\\'], destImageSys)

            # 如果发送成功，更新图像数据库，将sendFlag设置为1
            if ret:
                cursor.execute("UPDATE imageLevel SET sendFlag = 1 WHERE SOPInsUID = ?", (imageRecord['SOPInsUID'],))

        # 提交数据库更改
        connection.commit()

    except Exception as e:
        print(f"Error: {e}")

    finally:
        # 关闭数据库连接
        if connection:
            connection.close()


# 主程序启动时，设定图像发送定时器，触发imageSendProc函数
def imageSendProc():
    sqlite_DB_Filename = "pcas2023.db"
    destImageSys = "your_destination_system"

    # 设置定时器，例如每120秒一次
    while True:
        sendImagesToTargetSystem('pcas2023.db', '127.0.0.1')
        dicom_send_image(r'D:\Che Qiancao\sjtu\课程\7大四上\课程设计\项目设计_模块3_520021910466_仇清扬\\' + sopInsUID + 'modified_dicom_file1.dcm', 'SC_Capture', 'RADIANT', '127.0.0.1', 11112)
        time.sleep(120)



In [6]:
from pydicom import dcmread
from pynetdicom import AE, debug_logger
from pynetdicom.sop_class import CTImageStorage, MRImageStorage
from pydicom.uid import ImplicitVRLittleEndian, JPEGBaseline

def dicom_send_image(file_name, own_ae_title, peer_ae_title, peer_ip, peer_port):
    # Create an Application Entity with the specified AE title
    ae = AE(ae_title=own_ae_title)

    # Read the DICOM file to determine its SOP Class UID
    ds = dcmread(file_name)
    

    # Add a requested presentation context based on the SOP Class UID
    #sop_class = ds.SOPClassUID
    #ae.add_requested_context(sop_class)
    ae.add_requested_context(CTImageStorage)    
    
    # Associate with the peer AE at the specified IP and port using the specified AE title
    assoc = ae.associate(peer_ip, peer_port, ae_title=peer_ae_title)
    if assoc.is_established:
        # Send the DICOM dataset using the C-STORE service
        status = assoc.send_c_store(ds)

        # Check the status of the storage request
        if status:
            print('C-STORE request status: 0x{0:04x}'.format(status.Status))
        else:
            print('Connection timed out, was aborted or received invalid response')

        # Release the association
        assoc.release()
        return True
    else:
        print('Association rejected, aborted or never connected')
        return False
        

In [7]:
if __name__ == '__main__':
    timer_thread = threading.Thread(target=imageSendProc)
    timer_thread.daemon = True  # 设置为守护线程，主程序退出时线程也会退出
    timer_thread.start()
    app = QApplication(sys.argv)
    main_window = MainUI()
    main_window.show()
    sys.exit(app.exec_())

Exception in thread Thread-6:
Traceback (most recent call last):
  File "D:\Program_D\anaconda3\lib\threading.py", line 973, in _bootstrap_inner
    self.run()
  File "D:\Program_D\anaconda3\lib\threading.py", line 910, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\HUAWEI\AppData\Local\Temp\ipykernel_21592\310127441.py", line 43, in imageSendProc
NameError: name 'sopInsUID' is not defined


Error: no such column: sendFlag
{'Patient': {'PatNam': '4', 'PatBirDate': '1990-1-1', 'PatSex': 'M', 'Sex': 'M', 'Age': 34, 'NumPatRelStu': 0, 'NumPatRelSer': 0, 'NumPatRelIma': 0, 'PatID': '2.25.1.17048983735712'}, 'Study': {'StuInsUID': '2.25.2.17048983739515', 'OtherStudyInfo': 'Physician: ffff', 'StuID': 1, 'StuDate': '20240110', 'StuTime': '225253', 'AccNum': '1', 'PatAge': '34', 'PatSize': 170, 'PatWeight': 70, 'NumStuRelSer': 0, 'NumStuRelIma': 0}, 'Series': {'SerInsUID': '2.25.3.17048983736234', 'OtherSeriesInfo': 'Comments: aaaa', 'SerNum': '1', 'Modality': 'ES', 'ProNam': 'ffff', 'SerDes': 'aaaa', 'BodParExa': 'stomach', 'NumSerRelIma': 0}, 'Image': {'SOPInsUID': '', 'StoragePath': ''}}


D:\Program_D\anaconda3\lib\site-packages\pydicom\valuerep.py:443: UserWarning: Invalid value for VR DA: '1990-1-1'.
  warnings.warn(msg)
D:\Program_D\anaconda3\lib\site-packages\pydicom\valuerep.py:443: UserWarning: Invalid value for VR AS: '34'.
  warnings.warn(msg)
D:\Program_D\anaconda3\lib\site-packages\pydicom\valuerep.py:443: UserWarning: Invalid value for VR CS: 'stomach'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warnings.warn(msg)
D:\Program_D\anaconda3\lib\site-packages\pydicom\valuerep.py:443: UserWarning: The value length (177) exceeds the maximum length of 64 allowed for VR LO.
  warnings.warn(msg)
D:\Program_D\anaconda3\lib\site-packages\pydicom\valuerep.py:443: UserWarning: The value length (78) exceeds the maximum length of 64 allowed for VR LO.
  warnings.warn(msg)
D:\Program_D\anaconda3\lib\site-packages\pydicom\valuerep.py:443: UserWarning: The value length (256) exceeds the maximum

Dataset.file_meta -------------------------------
(0002, 0000) File Meta Information Group Length  UL: 164
(0002, 0001) File Meta Information Version       OB: b'\x00\x01'
(0002, 0002) Media Storage SOP Class UID         UI: Secondary Capture Image Storage
(0002, 0003) Media Storage SOP Instance UID      UI: 2.25.4.17048983739563
(0002, 0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002, 0012) Implementation Class UID            UI: 1.2.826.0.1.3680043.8.498.1
(0002, 0013) Implementation Version Name         SH: 'PYDICOM 2.4.4'
-------------------------------------------------
(0008, 0008) Image Type                          CS: ['DERIVED', 'SECONDARY']
(0008, 0018) SOP Instance UID                    UI: 2.25.4.17048983739563
(0008, 0020) Study Date                          DA: '20240110'
(0008, 0030) Study Time                          TM: '225253'
(0008, 0050) Accession Number                    SH: '1'
(0008, 0060) Modality                            CS: 

SystemExit: 0

D:\Program_D\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3406: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [25]:
import gradio as gr
import pydicom
import hashlib
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.backends import default_backend
import io
import datetime
from pydicom.dataset import Dataset
from pydicom import Sequence
import numpy as np
from cryptography.hazmat.primitives.serialization import load_pem_public_key
from Crypto.PublicKey import RSA


# Define a function to recursively extract DICOM attributes
def extract_dicom_attributes(dicom_dataset):
    attributes = []

    for element in dicom_dataset:
        attribute_name = element.description()
        attribute_value = str(element.value)

        if isinstance(element.value, pydicom.sequence.Sequence):
            for item in element.value:
                # Recursively extract attributes from sequences
                sub_attributes = extract_dicom_attributes(item)
                attributes.extend(sub_attributes)
        else:
            attributes.append((attribute_name, attribute_value))

    return attributes

# Define a function to read the DICOM file, extract attributes, and display them in a table
def read_dicom(file):
    dicom_data = pydicom.dcmread(file.name,force=True)
    dicom_file = pydicom.dcmread(file.name,force=True)
    pixel_data = dicom_data.pixel_array
    #chatgpt: how to use gradio.File
    origin_pixel_data = dicom_data.pixel_array
    changed_pixel_data = ((origin_pixel_data.astype(np.float32) / np.max(origin_pixel_data)-np.min(origin_pixel_data)) * 255).astype(np.uint8)
    
    flag=0
    # Extract DICOM attributes, including sequences
    attributes = extract_dicom_attributes(dicom_data)

    # Create a table-like Markdown string for DICOM attributes and their values
    markdown_info = "| Attribute | Value |\n"
    markdown_info += "| --- | --- |\n"

    for attribute_name, attribute_value in attributes:
        markdown_info += f"| {attribute_name} | {attribute_value} |\n"
        
    #计算
    with open(r'D:\\Che Qiancao\\sjtu\\课程\\7大四上\\课程设计\\lesson4\\private_key.pem', "rb") as key_file:
        public_key_data = key_file.read()
    if b"-----BEGIN PUBLIC KEY-----" not in public_key_data:
        # Assuming it's in DER format, convert to PEM
        public_key_data = serialization.load_der_public_key(public_key_data, backend=default_backend())
        public_key_bytes = public_key_data.public_bytes(
        encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        )
    else:
        public_key_bytes = public_key_data
    backend = openssl.backend
    public_key = serialization.load_pem_public_key(public_key_bytes, backend=backend)
    
    image_data = dicom_data.pixel_array
    gray_image_data = cv2.cvtColor(image_data, cv2.COLOR_BGR2GRAY)
    height, width = gray_image_data.shape
    
    
    top_left = image_data[:height // 2, :width // 2]
    top_right = image_data[:height // 2, width // 2:]
    bottom_left = image_data[height // 2:, :width // 2]
    bottom_right = image_data[height // 2:, width // 2:]
    
    def hash_ec(dicom_image):
        #hash 值
        sha256 = hashlib.sha256()
        sha256.update(dicom_image.tobytes())
        hash_value=sha256.digest()
        return hash_value
    
    new_tl=hash_ec(top_left)
    new_tr=hash_ec(top_right)
    new_bl=hash_ec(bottom_left)
    new_br=hash_ec(bottom_right)
    
    def hash_de(encrypted_hash,hash_value):
        signature_bytes = bytes(encrypted_hash)
        verification_result = public_key.verify(
            signature_bytes,
            hash_value,
            padding.PKCS1v15(),
            hashes.SHA256()
        )
        if verification_result  == None:
            check=0
        else:
            check=1
        return check
    

    # 提取数字签名序列
    signatures_sequence = dicom_file.get((0x9001, 0x1000), [])
    
    extracted_signatures = {}

    # 遍历每个签名数据集
    for i, signature_dataset in enumerate(signatures_sequence):
        # 获取签名区域信息
        auth_region = signature_dataset.get((0x9001, 0x0006), '')
        
        # 提取UL, UR, BL, BR 和 Attributes 签名
        if auth_region == 'UL':
            extracted_signatures['UL'] = {
                'signature': signature_dataset.get((0x9001, 0x0001), ''),
                'signature_time': signature_dataset.get((0x9001, 0x0002), '')
            }
        elif auth_region == 'UR':
            extracted_signatures['UR'] = {
                'signature': signature_dataset.get((0x9001, 0x0001), ''),
                'signature_time': signature_dataset.get((0x9001, 0x0002), '')
            }
        elif auth_region == 'BL':
            extracted_signatures['BL'] = {
                'signature': signature_dataset.get((0x9001, 0x0001), ''),
                'signature_time': signature_dataset.get((0x9001, 0x0002), '')
            }
        elif auth_region == 'BR':
            extracted_signatures['BR'] = {
                'signature': signature_dataset.get((0x9001, 0x0001), ''),
                'signature_time': signature_dataset.get((0x9001, 0x0002), '')
            }
        elif auth_region == 'Attributes':
            extracted_signatures['Attributes'] = {
                'signature': signature_dataset.get((0x9001, 0x0001), ''),
                'signature_time': signature_dataset.get((0x9001, 0x0002), '')
            }
    os_tl = extracted_signatures.get('UL', {})
    if(hash_de(os_tl,new_tl)==1):
        print("图片左上角被篡改了。")
        flag=1
    os_tr = extracted_signatures.get('UR', {})
    
    if(hash_de(os_tr,new_tr)==1):
        print("图片右上角被篡改了。")
        flag=1
    os_bl = extracted_signatures.get('BL', {})
    if(hash_de(os_bl,new_bl)==1):
        print("图片左下角被篡改了。")
        flag=1
    os_br = extracted_signatures.get('BR', {})
    if(hash_de(os_tr,new_br)==1):
        print("图片右下角被篡改了。")
        flag=1
    
    os_mega = extracted_signatures.get('Attributes', {})
    
    patient_name = dicom_data.PatientName
    patient_age = dicom_data.PatientAge


    new_str = f"{patient_name} {patient_age}"
    new_meg = private_key.sign(
            metadata_str.encode(),
            padding.PKCS1v15(),
            hashes.SHA256()
    )
    verification_result2 = public_key.verify(
            os_mega,
            new_meg,
            padding.PKCS1v15(),
            hashes.SHA256()
    )
    if(verification_result2!=None):
        print("文件的病人信息、年龄被篡改了。")

    return changed_pixel_data,markdown_info

# Create a Gradio interface with a File input component
file_input = gr.File(label="选择DICOM图像文件")

# GPT: how to use gradio.File to read a dicom and use gr.Markdown to show the information. The information includes name and value. The information should be like a table. When facing sequence , expand these data, use the iterative algorithm
gr.Interface(fn=read_dicom, inputs=file_input, 
    outputs=[gr.Image(label="处理后图像"),gr.Markdown()],
    title="DICOM图像篡改检测系统",  # 界面标题
    description="DICOM图像展示成表格，显示图像",  # 界面描述
    ).launch()

Running on local URL:  http://127.0.0.1:7872

To create a public link, set `share=True` in `launch()`.
